In [ ]:
import requests
from bs4 import BeautifulSoup
import csv
import time
import json

BASE_URL = "https://nanoreview.net"
LIST_URL = "https://nanoreview.net/en/phone-list/antutu-rating"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


def get_page(url, retries=3, delay=2):
    """Fetch a page with retry logic."""
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            print(f"  [!] Attempt {attempt + 1} failed for {url}: {e}")
            if attempt < retries - 1:
                time.sleep(delay)
    return None


def get_total_pages(soup):
    """Extract total number of pages from pagination."""
    pagination = soup.select_one(".pagination-map")
    if pagination:
        text = pagination.get_text()
        parts = text.split("of")
        if len(parts) > 1:
            pages = parts[1].strip().split()[0]
            return int(pages)
    return 1


def parse_table(soup):
    """Parse the benchmark table rows into a list of dicts."""
    rows = []
    table = soup.select_one("table.table-list tbody")
    if not table:
        return rows

    for tr in table.find_all("tr"):
        cols = tr.find_all("td")
        if len(cols) < 6:
            continue

        rank = cols[0].get_text(strip=True)

        phone_tag = cols[1].find("a")
        phone_name = phone_tag.get_text(strip=True) if phone_tag else cols[1].get_text(strip=True)
        phone_url = BASE_URL + phone_tag["href"] if phone_tag and phone_tag.get("href") else ""

        score_div = cols[2]
        bar = score_div.find("div", class_="score-bar-line")
        if bar:
            bar.decompose()
        antutu_score = score_div.get_text(strip=True)

        processor = cols[3].get_text(strip=True)
        gpu = cols[4].get_text(strip=True)
        ram = cols[5].get_text(strip=True)

        rows.append({
            "rank": rank,
            "phone": phone_name,
            "antutu_v11": antutu_score,
            "processor": processor,
            "gpu": gpu,
            "ram": ram,
            "url": phone_url,
        })

    return rows


def scrape_all_pages(max_pages=None):
    """Scrape all pages and return combined data."""
    print(f"Fetching page 1: {LIST_URL}")
    html = get_page(LIST_URL)
    if not html:
        print("Failed to fetch first page.")
        return []

    soup = BeautifulSoup(html, "html.parser")
    total_pages = get_total_pages(soup)
    if max_pages:
        total_pages = min(total_pages, max_pages)

    print(f"Total pages to scrape: {total_pages}")

    all_data = parse_table(soup)
    print(f"  Page 1: {len(all_data)} records")

    for page in range(2, total_pages + 1):
        url = f"{LIST_URL}?page={page}"
        print(f"Fetching page {page}: {url}")
        html = get_page(url)
        if not html:
            print(f"  Skipping page {page} (failed to fetch)")
            continue
        page_soup = BeautifulSoup(html, "html.parser")
        rows = parse_table(page_soup)
        print(f"  Page {page}: {len(rows)} records")
        all_data.extend(rows)
        time.sleep(1.5) 

    return all_data


def save_csv(data, filename="antutu_rankings.csv"):
    if not data:
        print("No data to save.")
        return
    fieldnames = ["rank", "phone", "antutu_v11", "processor", "gpu", "ram", "url"]
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(data)
    print(f"Saved {len(data)} records to {filename}")



if __name__ == "__main__":
    data = scrape_all_pages(max_pages=None)

    if data:
        print(f"\nTotal records scraped: {len(data)}")
        print("\nSample (first 3):")
        for item in data[:3]:
            print(f"  #{item['rank']} {item['phone']} — {item['antutu_v11']} "
                  f"({item['processor']}, {item['gpu']}, {item['ram']})")

        save_csv(data, "../GSMArenaDataset/antutu_score.csv")

    else:
        print("No data scraped.")

Fetching page 1: https://nanoreview.net/en/phone-list/antutu-rating
Total pages to scrape: 6
  Page 1: 200 records
Fetching page 2: https://nanoreview.net/en/phone-list/antutu-rating?page=2
  Page 2: 200 records
Fetching page 3: https://nanoreview.net/en/phone-list/antutu-rating?page=3
  Page 3: 200 records
Fetching page 4: https://nanoreview.net/en/phone-list/antutu-rating?page=4
  Page 4: 200 records
Fetching page 5: https://nanoreview.net/en/phone-list/antutu-rating?page=5
  Page 5: 200 records
Fetching page 6: https://nanoreview.net/en/phone-list/antutu-rating?page=6
  Page 6: 123 records

Total records scraped: 1123

Sample (first 3):
  #1 ZTE Nubia Red Magic 11S Pro — 4746315 (Snapdragon 8 Elite Gen 5, Adreno 840, 16 GB)
  #2 ZTE Nubia Red Magic 11S Pro Plus — 4592636 (Snapdragon 8 Elite Gen 5, Adreno 840, 16 GB)
  #3 Honor Magic 8 Pro Air — 4430255 (Dimensity 9500, Mali-G1 Ultra MP12, 16 GB)


FileNotFoundError: [Errno 2] No such file or directory: 'GSMArenaDataset/antutu_score.csv'